# NIR Looks Like Spaghetti — SNV and MSC Untangle It
### 4.3 — NIR Preprocessing: SNV, MSC, and Scatter Correction

Overlay a few dozen raw near-infrared (NIR) spectra of *related* samples and you
get a plate of spaghetti: broad bands smeared across the axis, every spectrum
shifted and tilted differently, the chemistry nowhere to be seen. That mess is
not bad instrumentation — it is **physical light scatter**, and it is the single
biggest obstacle between a raw NIR spectrum and a useful number.

This notebook is **not** "call `snv()` and move on." It builds the reasoning
around scatter correction:

- why NIR spectra are **scatter-dominated** in the first place;
- how **multiplicative** (slope) and **additive** (offset) scatter distort a
  spectrum, and why that makes raw overlays look like spaghetti;
- what **SNV** does — mathematically and visually;
- what **MSC** does — mathematically and visually;
- **when** SNV and MSC help, and **when they quietly hurt**;
- how scatter correction changes **PCA / PLS readiness**;
- how to **grade** a correction against the *known* per-sample scatter, because
  the data is simulated and carries its own ground truth.

> **The one idea to hold onto.** SNV and MSC are **not** universally good
> preprocessing steps. They are **assumptions about your data** — specifically,
> that the dominant sample-to-sample variation is *nuisance scatter*. The
> question you must answer before applying them is always the same: **is the
> variation I'm about to remove physical scatter, or is it chemically meaningful
> signal?** Get that wrong and you will erase the very thing you came to measure.
> We will make that failure happen on purpose, with numbers, near the end.

## 1. Setup

Just the standard scientific-Python stack plus our project's NIR simulator. We do
PCA later with NumPy's SVD — no `scikit-learn` — to keep the machinery visible
rather than hidden behind a black box.

Because the spectra come from `nir.simulate()`, **we know the true per-sample
scatter** (its slope and offset) and the **true property** behind each sample.
That is the answer key real data can never give us, and it is what lets us *grade*
every correction instead of just admiring it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Our NIR simulator. Simulated data carries its own ground truth (true scatter
# slope/offset per sample), so we can check honestly whether a correction
# removed scatter -- or removed chemistry.
from simulated_data import nir

# Figures are regenerable scratch; the exports/ folder is git-ignored.
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

SEED = 4                      # fix every random draw so your output matches ours
plt.rcParams["figure.dpi"] = 110

## 2. Why NIR spectra are scatter-dominated

Three facts about NIR conspire to bury the chemistry:

1. **The bands are broad and overlapping.** NIR absorptions are *overtones* and
   *combinations* of fundamental mid-IR vibrations — weak, wide, and piled on top
   of each other. There are no sharp, isolated peaks to read; the chemical
   information is spread thinly across the whole axis.

2. **The measurement is diffuse reflectance.** You usually shine light at a
   powder, slurry, or tablet and collect what scatters back. How much light
   returns depends on **particle size, packing density, and surface texture** —
   the *physical presentation* of the sample — not just its chemistry.

3. **Presentation varies sample to sample.** Grind one tablet finer, pack one
   cup tighter, and the path the light takes changes. The result is the standard,
   and remarkably accurate, scatter model: each measured spectrum is its true
   absorbance **multiplied by a per-sample slope and shifted by a per-sample
   offset**:

$$x_{\text{measured}}(\lambda) \;=\; b \cdot x_{\text{true}}(\lambda) \;+\; a$$

The multiplicative term $b$ is the path-length / scattering-efficiency change;
the additive term $a$ is a baseline shift. **Both are nuisance** — they carry no
chemistry — yet they dominate the raw signal. Removing them is what SNV and MSC
are for.

## 3. A realistic NIR set with known ground truth

We build 24 samples that share the same broad NIR matrix (three fixed
overlapping bands) but differ in **one diagnostic analyte band** whose height
encodes a property `y` (think moisture, or API content). This is a deliberate
design choice:

- the chemistry is a **band-ratio / shape** change — the analyte band grows
  relative to the fixed matrix. A *shape* change is exactly the kind of signal a
  good scatter correction should **preserve**;
- on top of that, every sample gets its **own** multiplicative slope and additive
  offset (the scatter) plus detector noise.

`nir.simulate()` applies and **records** the true scatter per sample, so we can
grade ourselves later. One shared random generator threads through every call, so
the whole set reproduces from `SEED`.

In [ ]:
def make_nir_set(n_samples=24, mode="band_ratio", scatter=True,
                 noise=0.003, seed=SEED):
    """Build an NIR set whose per-sample property we know exactly.

    The property `y` is the ground-truth chemistry. Two modes:

    * 'band_ratio'  -> y sets the height of one diagnostic analyte band against a
      fixed broad matrix. Chemistry is a SHAPE change, which honest scatter
      correction should PRESERVE. (Our primary dataset.)
    * 'global_level'-> y scales the whole spectrum. Chemistry is an overall LEVEL,
      which SNV/MSC will normalize AWAY. (Used later for the 'when it hurts' demo.)

    Each sample also gets its own scatter slope + offset (drawn and recorded by
    nir.simulate, read back from meta) and detector noise. A single shared RNG is
    threaded through every call, so the whole set is reproducible from `seed`.
    """
    rng = np.random.default_rng(seed)

    # The true property, then shuffled so a sample's ROW INDEX is unrelated to its
    # property -- nothing downstream can cheat off ordering.
    y = np.linspace(0.2, 0.9, n_samples)
    y = y[rng.permutation(n_samples)]

    # Three broad, overlapping matrix bands shared by every sample: (nm, FWHM, amp).
    matrix = [(1450.0, 130.0, 0.70), (1930.0, 150.0, 0.90), (2250.0, 200.0, 0.55)]

    rows, slope, offset = [], [], []
    for p in y:
        if mode == "band_ratio":
            # Diagnostic analyte band at 1720 nm; its HEIGHT encodes the property.
            peaks = matrix + [(1720.0, 110.0, 0.15 + 0.55 * p)]
            conc = 1.0
        else:  # global_level: fixed shape, property scales the whole spectrum.
            peaks = matrix + [(1720.0, 110.0, 0.30)]
            conc = 0.6 + 0.8 * p

        ds = nir.simulate(n_samples=1, peaks=peaks, concentration=conc,
                          scatter=scatter, noise=noise, seed=rng, n_points=700)
        rows.append(ds.X[0])
        slope.append(ds.meta["scatter_slope"].iloc[0])
        offset.append(ds.meta["scatter_offset"].iloc[0])

    return ds.x, np.vstack(rows), y, np.array(slope), np.array(offset)


wl, X, y, true_slope, true_offset = make_nir_set(mode="band_ratio")
print("X shape       :", X.shape)
print("property range : %.2f to %.2f" % (y.min(), y.max()))
print("scatter slope  : %.3f +/- %.3f (truth, per sample)" % (true_slope.mean(), true_slope.std()))
print("scatter offset : %.3f +/- %.3f (truth, per sample)" % (true_offset.mean(), true_offset.std()))

### The answer key

Because we simulated the data, the true scatter behind each spectrum is on the
record. Here are the first few samples: the property we want to recover, and the
nuisance scatter we want to remove.

In [ ]:
truth = pd.DataFrame(
    {"property_y": y, "scatter_slope": true_slope, "scatter_offset": true_offset}
).round(3)
truth.head(6)

## 4. The spaghetti plot

Here is the raw set, every spectrum colored by its **true property**. If the
chemistry were visible, the colors would stack in a smooth gradient. Instead the
spectra are shifted and tilted all over each other — the color order is scrambled.
**The scatter, not the chemistry, controls what you see.**

In [ ]:
ANALYTE_NM = 1720.0
ai = int(np.argmin(np.abs(wl - ANALYTE_NM)))   # index of the analyte band peak

cmap = plt.cm.viridis
norm = plt.Normalize(y.min(), y.max())

fig, ax = plt.subplots(figsize=(9, 4.6))
for i in range(X.shape[0]):
    ax.plot(wl, X[i], color=cmap(norm(y[i])), lw=1.0, alpha=0.9)
ax.axvline(ANALYTE_NM, color="0.6", ls=":", lw=1)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Absorbance")
ax.set_title("Raw NIR spectra, colored by the TRUE property  -  spaghetti")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=ax, label="property y (true)")
fig.tight_layout()
fig.savefig(EXPORTS / "01_raw_spaghetti.png")
plt.show()

## 5. Anatomy of scatter: multiplicative vs. additive

Take a single clean spectrum (no scatter, no noise) — the underlying chemistry —
and apply the two effects separately. A **multiplicative** slope ($\times b$)
stretches the whole spectrum taller or shorter; an **additive** offset ($+a$)
lifts it bodily off the baseline. Real scatter is both at once. None of it
touches the *positions* of the bands — only their apparent scale and level — which
is precisely why a per-spectrum rescale-and-shift can undo it.

In [ ]:
# One clean spectrum = the true chemistry, with scatter and noise switched off.
_, clean_set, _, _, _ = make_nir_set(n_samples=1, scatter=False, noise=0.0, seed=99)
clean = clean_set[0]

mult = 1.30 * clean            # pure multiplicative (slope): taller everywhere
add  = clean + 0.25            # pure additive (offset): shifted up bodily
both = 1.30 * clean + 0.25     # both together = what real scatter does

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(wl, clean, color="black", lw=2.2, label="clean (true chemistry)")
ax.plot(wl, mult, lw=1.4, label=r"$\times\,b$  multiplicative (slope)")
ax.plot(wl, add,  lw=1.4, label=r"$+\,a$  additive (offset)")
ax.plot(wl, both, lw=1.6, ls="--", label=r"$b\cdot x + a$  both (real scatter)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Absorbance")
ax.set_title("How multiplicative and additive scatter distort one spectrum")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "02_scatter_anatomy.png")
plt.show()

## 6. SNV — Standard Normal Variate

SNV is the simplest scatter correction, and it is **reference-free**: each
spectrum is standardized using only itself. Subtract that spectrum's own mean,
divide by its own standard deviation:

$$x_{\text{SNV}}(\lambda) \;=\; \frac{x(\lambda) - \bar{x}}{s_x}$$

where $\bar{x}$ and $s_x$ are the mean and standard deviation **across
wavelengths** of that one spectrum. Subtracting $\bar{x}$ removes the additive
offset $a$; dividing by $s_x$ removes the multiplicative slope $b$. After SNV
every spectrum has mean 0 and standard deviation 1.

**The assumption baked in:** that $\bar{x}$ and $s_x$ are set by *scatter*. If a
chemical change also moves a spectrum's mean or spread, SNV will partially remove
*that* too. We keep the analyte band modest here so scatter dominates the
statistics — and we test exactly this assumption later.

In [ ]:
def snv(X):
    """Standard Normal Variate: standardize each spectrum on its own.

    Row-wise: subtract each spectrum's mean (kills the additive offset) and divide
    by its standard deviation (kills the multiplicative slope). Reference-free --
    no calibration set needed, which is why SNV is so widely used.
    """
    mu = X.mean(axis=1, keepdims=True)     # per-spectrum mean  (n_samples, 1)
    sd = X.std(axis=1, keepdims=True)      # per-spectrum std   (n_samples, 1)
    return (X - mu) / sd


X_snv = snv(X)
print("after SNV: overall mean ~", round(float(X_snv.mean()), 6),
      "| per-spectrum std ~", round(float(X_snv.std(axis=1).mean()), 3))

## 7. MSC — Multiplicative Scatter Correction

MSC asks a slightly different question: *how does this spectrum differ, by a
slope and an offset, from a typical spectrum?* It fits each spectrum to a
**reference** — usually the mean spectrum of the set — by least squares:

$$x(\lambda) \;\approx\; a \;+\; b\cdot x_{\text{ref}}(\lambda)$$

The fitted intercept $a$ and slope $b$ **are** the additive and multiplicative
scatter, so the corrected spectrum removes them:

$$x_{\text{MSC}}(\lambda) \;=\; \frac{x(\lambda) - a}{b}$$

The crucial difference from SNV: **MSC needs a reference.** That reference is part
of the model — fit it on your calibration set and you must reuse *the same*
reference on every future sample. This makes MSC **reference-dependent** (a
liability for instrument/calibration transfer), whereas SNV depends on nothing but
the spectrum in front of it.

In [ ]:
def msc(X, reference=None):
    """Multiplicative Scatter Correction against a reference spectrum.

    Model each spectrum as  x ~ a + b * reference  by ordinary least squares; the
    fitted (a, b) are the additive and multiplicative scatter, so the corrected
    spectrum is (x - a) / b. If no reference is given, use the mean spectrum --
    but keep it: new samples must be corrected against the SAME reference.
    """
    if reference is None:
        reference = X.mean(axis=0)
    # Design matrix columns [1, reference]; lstsq returns (a, b) per spectrum.
    design = np.vstack([np.ones_like(reference), reference]).T
    out = np.empty_like(X)
    for i, xi in enumerate(X):
        (a, b), *_ = np.linalg.lstsq(design, xi, rcond=None)
        out[i] = (xi - a) / b
    return out, reference


X_msc, msc_ref = msc(X)
print("MSC reference spectrum kept for reuse, shape:", msc_ref.shape)

## 8. Raw vs. SNV vs. MSC

The same 24 spectra, three ways, each colored by the true property. The raw panel
is spaghetti. After SNV or MSC the spectra collapse onto a common scale and the
**color gradient appears** — the analyte band near 1720 nm now fans out cleanly
with the property. SNV and MSC give visually very similar results here; that is
typical when the scatter is well described by a single slope and offset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, M, title in [
    (axes[0], X,     "Raw (scatter-dominated)"),
    (axes[1], X_snv, "SNV corrected"),
    (axes[2], X_msc, "MSC corrected"),
]:
    for i in range(M.shape[0]):
        ax.plot(wl, M[i], color=cmap(norm(y[i])), lw=0.9, alpha=0.9)
    ax.axvline(ANALYTE_NM, color="0.6", ls=":", lw=1)
    ax.set_title(title)
    ax.set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Absorbance / standardized")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=axes, label="property y (true)", fraction=0.025, pad=0.01)
fig.suptitle("Scatter correction reveals the chemistry that raw overlays hide", y=1.02)
fig.savefig(EXPORTS / "03_raw_snv_msc.png", bbox_inches="tight")
plt.show()

## 9. Did it actually work? Grade against the known scatter

Eyeballing is not enough. Because the truth is in hand, we can measure two things:

1. **Scatter leakage into the raw signal.** How strongly does each raw spectrum's
   *mean* track the true additive offset, and its *spread* the true
   multiplicative slope? If those correlations are high, the raw data really is
   scatter-dominated.
2. **Chemistry recovery.** How strongly does the **analyte band height** track the
   true property — before vs. after correction? A good correction *raises* this.

In [ ]:
def corr(a, b):
    return float(np.corrcoef(a, b)[0, 1])

print("Is the RAW signal dominated by scatter?")
print(f"  raw spectrum mean  vs true offset : {corr(X.mean(1), true_offset):+.3f}")
print(f"  raw spectrum std   vs true slope  : {corr(X.std(1),  true_slope):+.3f}")

print()
print("Does the analyte-band height track the PROPERTY (chemistry)?")
for name, M in [("raw", X), ("SNV", X_snv), ("MSC", X_msc)]:
    print(f"  {name:3s} : {corr(M[:, ai], y):+.3f}")

## 10. How scatter correction changes PCA / PLS readiness

PCA finds the directions of **largest variance**. On raw NIR that largest variance
is *scatter*, so PCA spends its first component describing nuisance — not
chemistry. Scatter correction removes that nuisance, freeing the leading component
to capture the property. This is exactly what a downstream **PLS** model needs:
predictor variation aligned with the thing you want to predict, not with how
tightly someone packed the sample cup.

We run PCA from scratch (mean-center, then SVD) and plot the scores twice per
method — colored by the **true property** and by the **true scatter offset** — and
print how the first component correlates with each. Watch the contrast: on **raw**
data the property hides in PC2 while the scatter offset orders PC1; after
correction the property *takes over* PC1 and the scatter structure is gone.

In [ ]:
def pca_scores(M, k=2):
    # Mean-center then SVD; return the first k scores and their variance share.
    Mc = M - M.mean(axis=0)
    U, S, Vt = np.linalg.svd(Mc, full_matrices=False)
    scores = U[:, :k] * S[:k]
    var = (S ** 2) / (S ** 2).sum()
    return scores, var[:k]

methods = [("Raw", X), ("SNV", X_snv), ("MSC", X_msc)]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (name, M) in enumerate(methods):
    sc, var = pca_scores(M)
    for row, (color_by, label, cm) in enumerate([
        (y, "property y", plt.cm.viridis),
        (true_offset, "true scatter offset", plt.cm.plasma),
    ]):
        ax = axes[row, col]
        sp = ax.scatter(sc[:, 0], sc[:, 1], c=color_by, cmap=cm, s=45,
                        edgecolor="k", linewidth=0.3)
        ax.set_xlabel(f"PC1 ({100*var[0]:.0f}%)")
        ax.set_ylabel(f"PC2 ({100*var[1]:.0f}%)")
        if row == 0:
            ax.set_title(name)
        fig.colorbar(sp, ax=ax, label=label, fraction=0.046, pad=0.04)
    print(f"{name:3s}: PC1 var {100*var[0]:4.1f}%  | "
          f"PC1~property {corr(sc[:,0], y):+.2f}  "
          f"PC1~scatter_slope {corr(sc[:,0], true_slope):+.2f}  "
          f"PC1~scatter_offset {corr(sc[:,0], true_offset):+.2f}")
fig.suptitle("PCA: raw PC1 tracks scatter; after correction PC1 tracks chemistry", y=1.0)
fig.tight_layout()
fig.savefig(EXPORTS / "04_pca_readiness.png", bbox_inches="tight")
plt.show()

## 11. When SNV and MSC help

They earn their keep when the dominant sample-to-sample variation really is
physical scatter, and the chemistry lives in band **shape / ratio** rather than
overall level:

- diffuse-reflectance NIR of **powders, granules, tablets, slurries** — anything
  where particle size and packing vary;
- you care about **which** bands are present and their **relative** sizes
  (composition, identity, a property encoded in band shape);
- you are about to feed the spectra to **PCA / PLS** and want the components to
  describe chemistry, not sample presentation.

In our band-ratio set both corrections turned an analyte-vs-property correlation
of about **0.79** into roughly **1.00**, and moved the chemistry from PC2 into a
dominant PC1. That is the win.

## 12. When they can hurt

Here is the part too many tutorials skip. **SNV and MSC are assumptions, not
defaults.** Each one *defines* the per-spectrum mean and scale (SNV), or the fit
to a reference (MSC), as nuisance and throws it away. If your real signal lives in
exactly that quantity, you will delete it.

We make it happen. Build a set where the property is the **overall level** of the
spectrum (e.g. total absorbance rising with concentration) under only mild
scatter. Raw total absorbance tracks the property strongly — and then SNV and MSC,
which remove each spectrum's level by construction, **destroy** it.

In [ ]:
wl2, Xg, yg, sg, og = make_nir_set(
    mode="global_level",
    scatter={"slope_sigma": 0.02, "offset_sigma": 0.03},   # mild scatter
    seed=4,
)
Xg_snv = snv(Xg)
Xg_msc, _ = msc(Xg)

print("Property here lives in the OVERALL LEVEL of the spectrum.")
print("Correlation of total absorbance with the property:")
print(f"  raw : {corr(Xg.sum(1), yg):+.3f}   <- the real signal is clearly there")
print(f"  SNV : {corr(Xg_snv.sum(1), yg):+.3f}   <- removed by standardizing each spectrum")
print(f"  MSC : {corr(Xg_msc.sum(1), yg):+.3f}   <- removed by dividing out each slope/offset")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].scatter(yg, Xg.sum(1), c=yg, cmap="viridis", edgecolor="k", linewidth=0.3)
axes[0].set_title(f"RAW: total absorbance vs property (r = {corr(Xg.sum(1), yg):+.2f})")
axes[1].scatter(yg, Xg_snv.sum(1), c=yg, cmap="viridis", edgecolor="k", linewidth=0.3)
axes[1].set_title(f"SNV: signal erased (r = {corr(Xg_snv.sum(1), yg):+.2f})")
for ax in axes:
    ax.set_xlabel("true property y")
    ax.set_ylabel("total absorbance (sum)")
fig.suptitle("When the chemistry IS the level, scatter correction throws it away", y=1.02)
fig.tight_layout()
fig.savefig(EXPORTS / "05_when_it_hurts.png", bbox_inches="tight")
plt.show()

The lesson is not "MSC is broken" — the corrections did exactly what they promise.
The lesson is that **the correction encodes a belief about your data**. Before
applying SNV or MSC, ask: *is the variation I am about to remove nuisance scatter,
or chemically meaningful signal?* If the answer is "I'm not sure," find out before
you normalize — compare a model with and without the step, and check it against a
property you can measure.

## 13. A quantitative scorecard

Pulling the band-ratio results into one table makes the trade-off legible: how
much variance PC1 captures, how well the analyte band tracks the property, and how
much scatter still leaks into PC1.

In [ ]:
def scorecard(M):
    sc, var = pca_scores(M)
    return {
        "PC1 variance %": round(100 * var[0], 1),
        "analyte ~ property": round(corr(M[:, ai], y), 3),
        "PC1 ~ scatter_offset": round(corr(sc[:, 0], true_offset), 3),
    }

card = pd.DataFrame({name: scorecard(M) for name, M in
                     [("Raw", X), ("SNV", X_snv), ("MSC", X_msc)]}).T
card

## 14. Reusable `snv()` and `msc()` — and the one rule for new data

Both functions above are written to drop into later notebooks (4.7 preprocessing
pipeline, 6.3 PCA, 6.5 PLS). The only subtlety is MSC's reference: **fit it on
your calibration set and reuse the *same* reference on every new sample** — never
recompute it from a fresh batch, or you change the model. SNV has no such
memory; it treats each spectrum independently.

In [ ]:
# A brand-new sample arrives later. SNV needs nothing but the spectrum:
_, new_set, new_y, _, _ = make_nir_set(n_samples=1, mode="band_ratio", seed=2025)
new_spectrum = new_set                       # shape (1, n_points)

new_snv = snv(new_spectrum)                   # reference-free
new_msc, _ = msc(new_spectrum, reference=msc_ref)   # REUSE the stored reference

print("new sample corrected:",
      "SNV", new_snv.shape, "| MSC", new_msc.shape,
      "| MSC reused the calibration reference:", new_msc.shape == new_spectrum.shape)

## Key Takeaways

- **NIR is scatter-dominated.** Broad overlapping overtone bands measured in
  diffuse reflectance means the largest variation between raw spectra is usually
  *physical scatter* — a per-sample multiplicative slope and additive offset — not
  chemistry. Raw overlays look like spaghetti for this reason.
- **SNV** standardizes each spectrum on its own (subtract its mean, divide by its
  std). Reference-free, trivial to apply, no calibration memory.
- **MSC** fits each spectrum to a reference ($x \approx a + b\,x_{\text{ref}}$) and
  divides out the fitted slope and offset. Reference-**dependent**: you must keep
  and reuse the reference.
- Here, both turned a hidden analyte signal (analyte~property ≈ 0.79 raw) into a
  clean one (≈ 1.00) and moved the chemistry onto **PC1**, readying the data for
  PCA/PLS.
- **They are assumptions, not defaults.** SNV/MSC define the per-spectrum level
  and scale as nuisance. When the chemistry actually *is* the level, they erase
  it — we watched a raw correlation of 0.87 collapse to near 0 after correction.
- **The deciding question is always:** *is the variation I'm removing nuisance
  scatter, or chemically meaningful signal?*

## Practical Checklist

- [ ] **Look at the raw overlay first.** Spaghetti (shifted, tilted spectra) is
      the signature that a scatter correction may help.
- [ ] **Name what varies.** Is your property a band **shape/ratio** (good
      candidate for SNV/MSC) or an overall **level** (danger — you may normalize
      it away)?
- [ ] **Pick one method deliberately.** SNV if you want a reference-free,
      transfer-friendly step; MSC if a stable mean spectrum is meaningful and you
      will manage the reference.
- [ ] **For MSC, store the reference** from the calibration set and reuse it on
      all new data.
- [ ] **Check it helped, don't assume.** Compare a downstream model (PCA spread,
      PLS error) with and without the step.
- [ ] **Order matters.** Decide where scatter correction sits relative to
      derivatives, smoothing, and mean-centering — and keep that order fixed
      between calibration and prediction (a topic for the 4.7 pipeline lesson).

## Common Mistakes

- **Applying SNV/MSC reflexively** because "everyone does it for NIR," without
  checking whether scatter is actually the dominant variation.
- **Normalizing away the signal.** Using SNV/MSC when the property is an overall
  intensity/level — the correction removes exactly that.
- **Recomputing the MSC reference on each new batch.** That silently changes the
  model; predictions stop being comparable. Fit once, reuse forever.
- **Trusting a pretty corrected overlay.** Aligned spectra look reassuring even
  when the correction hurt the model. Judge by a measured outcome, not the plot.
- **Forgetting preprocessing is part of the model.** Whatever you do to the
  calibration spectra you must do, identically, to every future spectrum.

## Reporting Guidance

When you publish or hand off an NIR method, state the preprocessing explicitly so
it is reproducible:

- **Which** correction (SNV or MSC) and, for MSC, **what reference** (e.g. "mean
  of the calibration set, n = …") and that it is reused at prediction time.
- **The full preprocessing order** (e.g. "MSC → 2nd-derivative Savitzky–Golay
  (window …, poly …) → mean-center").
- **Why** the step is justified for these samples (scatter from variable particle
  size / packing) — not merely "standard practice."
- **Evidence it helped**: the model metric with and without the step, on held-out
  data.
- The **wavelength range** used, since SNV/MSC statistics depend on which points
  are included.

## Next Lesson

**4.7 — Building an NIR Preprocessing Pipeline.** SNV and MSC are one stage. Real
NIR workflows chain scatter correction with **derivatives** (Savitzky–Golay, from
3.2) and mean-centering, and the **order** of those steps changes the result. Next
we assemble the full pipeline, fix the order, and validate it against ground truth
— then carry it into **6.3 (PCA)** and **6.5 (PLS)**, where the readiness we
measured here turns into a predictive model.